In [ ]:
"""
Galaxy-merger morphology analysis
=================================

The script creates a figures/ directory containing PDF figures (vector

graphics) and a CSV summary of the late-time measurements.

Column mapping follows the current problem.c output:

  1-17  global quantities

  18-21 vertical RMS in four radial bins

  22-25 A1 in four radial bins

  26-29 whole-primary/core COM offsets (x, y, z, magnitude)

  30    global A2

  31-34 A2 in four radial bins

Important: results_45deg.txt is used as the shared fiducial run:

i=45 deg, mass ratio 1:10, rp=2.5.

"""

from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------

# 1. FILES AND EXPERIMENT GROUPS

# ---------------------------------------------------------------------


# In a .py script, use the script's folder. In a Jupyter notebook,

# __file__ is not defined, so use the notebook's current working directory.

try:

    DATA_DIR = Path(__file__).resolve().parent

except NameError:

    DATA_DIR = Path.cwd()

# If your data files are in a different folder, replace the line above with:

# DATA_DIR = Path(r"/home/noro113/rebound/examples/selfgravity_disc")

OUTPUT_DIR = DATA_DIR / "figures"

OUTPUT_DIR.mkdir(exist_ok=True)

FILES = {

    # Inclination sweep: q=1:10, rp=2.5

    "inc_0":   "results_0deg.txt",

    "fiducial": "results_45deg.txt",

    "inc_90":  "results_90deg.txt",

    # Mass-ratio sweep: i=45 deg, rp=2.5

    "mass_1_5":  "results_1,5_Mass.txt",

    "mass_1_20": "results_1,20_Mass.txt",

    # Pericentre sweep: i=45 deg, q=1:10

    "rp_1_5": "results_1.5_b_param.txt",

    "rp_3_5": "results_3.5_b_param.txt",

}

# The fiducial run is deliberately reused in all three comparisons.

GROUPS = {

    "inclination": [

        ("inc_0", r"$i=0^\circ$"),

        ("fiducial", r"$i=45^\circ$ (fiducial)"),

        ("inc_90", r"$i=90^\circ$"),

    ],

    "mass_ratio": [

        ("mass_1_20", r"$1:20$"),

        ("fiducial", r"$1:10$ (fiducial)"),

        ("mass_1_5", r"$1:5$"),

    ],

    "pericentre": [

        ("rp_1_5", r"$r_p=1.5$"),

        ("fiducial", r"$r_p=2.5$ (fiducial)"),

        ("rp_3_5", r"$r_p=3.5$"),

    ],

}

# Late-time interval used for the summary statistics.

# Change this if your report uses a different settled interval.

SUMMARY_TMIN = 70.0

SUMMARY_TMAX = 100.0

# ---------------------------------------------------------------------

# 2. COLUMN MAP (ONE-BASED COLUMN NUMBERS FROM THE C OUTPUT)

# ---------------------------------------------------------------------

COL = {

    "time": 1,

    "N_primary_present": 2,

    "f_primary_present": 3,

    "N_core": 4,

    "f_core": 5,

    "R_rms_all": 6,

    "z_rms_all": 7,

    "R_rms_core": 8,

    "z_rms_core": 9,

    "z_core_cm": 10,

    "energy": 11,

    "Lz_all": 12,

    "x_cm": 13,

    "z_cm": 14,

    "deltaR_mean_abs": 15,

    "deltaR_rms": 16,

    "A1_all": 17,

    "z_rms_0_1": 18,

    "z_rms_1_2": 19,

    "z_rms_2_3": 20,

    "z_rms_3_4.25": 21,

    "A1_0_1": 22,

    "A1_1_2": 23,

    "A1_2_3": 24,

    "A1_3_4.25": 25,

    "dx_cm_core": 26,

    "dy_cm_core": 27,

    "dz_cm_core": 28,

    "dcm_core": 29,

    "A2_all": 30,

    "A2_0_1": 31,

    "A2_1_2": 32,

    "A2_2_3": 33,

    "A2_3_4.25": 34,

}

RADIAL_BINS = [
    ("0_1", r"$0 \leq R < 1$"),
    ("1_2", r"$1 \leq R < 2$"),
    ("2_3", r"$2 \leq R < 3$"),
    ("3_4.25", r"$3 \leq R < 4.25$"),
]

# ---------------------------------------------------------------------

# 3. LOAD AND VALIDATE ALL AVAILABLE FILES

# ---------------------------------------------------------------------

DATA = {}

missing = []

for key, filename in FILES.items():

    path = DATA_DIR / filename

    if not path.exists():

        missing.append(filename)

        continue

    arr = np.loadtxt(path, comments="#", ndmin=2)

    if arr.shape[1] != 34:

        raise ValueError(

            f"{filename} has {arr.shape[1]} columns; expected 34 based on "

            "the current problem_fixed_v2.c output. Check that the C source "

            "and the data files use the same column layout."

        )

    if not np.all(np.isfinite(arr[:, 0])):

        raise ValueError(f"{filename} contains invalid time values.")



    # Sort by time in case rows were not written in chronological order.

    arr = arr[np.argsort(arr[:, 0])]

    DATA[key] = arr



if missing:

    raise FileNotFoundError(

        "These expected data files were not found in "

        f"{DATA_DIR}:\n  - " + "\n  - ".join(missing)

    )



def col(data, name):

    """Return a data column using the one-based COL mapping."""

    return data[:, COL[name] - 1]



def save_figure(fig, filename):

    """Save a clean vector PDF and close the figure."""

    fig.savefig(OUTPUT_DIR / filename, format="pdf", bbox_inches="tight")

    plt.close(fig)

# ---------------------------------------------------------------------

# 4. GENERIC TIME-SERIES PLOTS FOR EACH EXPERIMENT SWEEP

# ---------------------------------------------------------------------

def plot_comparison(group_name, specs, metric, ylabel, filename,

                    ylim=None, zero_invalid=False):

    fig, ax = plt.subplots(figsize=(8.2, 5.0))



    for key, label in specs:

        data = DATA[key]

        t = col(data, "time")

        y = col(data, metric).copy()



        # Empty radial bins and invalid core diagnostics are marked -1.

        if zero_invalid:

            y[y < 0] = np.nan



        ax.plot(t, y, linewidth=1.8, label=label)



    ax.set_xlabel(r"Time, $t$ [simulation units]")

    ax.set_ylabel(ylabel)

    ax.set_xlim(left=0)

    if ylim is not None:

        ax.set_ylim(*ylim)

    ax.grid(True, alpha=0.25)

    ax.legend(frameon=False)

    fig.tight_layout()

    save_figure(fig, filename)



# Metrics that should be compared across all three parameter sweeps.

GLOBAL_METRICS = [

    ("f_primary_present", r"Fraction of original primary particles retained",

     "particle_retention", (0, 1.03), False),

    ("f_core", r"Core fraction, $f_{\rm core}$",

     "core_fraction", (0, 1.03), False),

    ("R_rms_all", r"Global radial RMS, $R_{\rm rms}$ [length units]",

     "global_radial_extent", None, False),

    ("z_rms_all", r"Global vertical RMS, $z_{\rm rms}$ [length units]",

     "global_vertical_extent", None, False),

    ("R_rms_core", r"Core radial RMS, $R_{\rm rms,core}$ [length units]",

     "core_radial_extent", None, True),

    ("z_rms_core", r"Core vertical RMS, $z_{\rm rms,core}$ [length units]",

     "core_vertical_extent", None, True),

    ("deltaR_mean_abs", r"Mean absolute radial displacement, "

     r"$\langle|\Delta R|\rangle$ [length units]",

     "mean_absolute_radial_migration", None, False),

    ("deltaR_rms", r"RMS radial displacement, $\Delta R_{\rm rms}$ "

     r"[length units]",

     "rms_radial_migration", None, False),

    ("A1_all", r"Global $A_1$",

     "global_A1", (0, 1), False),

    ("A2_all", r"Global $A_2$",

     "global_A2", (0, 1), False),

    ("dcm_core", r"Whole-primary/core COM separation, $d_{\rm CM}$ "

     r"[length units]",

     "core_whole_disk_COM_separation", None, True),

]



for group_name, specs in GROUPS.items():

    for metric, ylabel, suffix, ylim, invalid in GLOBAL_METRICS:

        plot_comparison(

            group_name, specs, metric, ylabel,

            f"{group_name}_{suffix}.pdf",

            ylim=ylim, zero_invalid=invalid

        )



# ---------------------------------------------------------------------

# 5. RADIAL-BIN PLOTS: VERTICAL THICKNESS, A1, AND A2

# ---------------------------------------------------------------------



def plot_radial_bins(group_name, specs, prefix, ylabel, filename,
                     ylim=None):
    # One panel per radial bin; each panel compares the three simulation runs.
    fig, axes = plt.subplots(
        2, 2, figsize=(10.0, 7.2), sharex=True, sharey=True
    )
    axes = axes.ravel()

    for ax, (bin_key, bin_label) in zip(axes, RADIAL_BINS):
        for key, run_label in specs:
            data = DATA[key]
            t = col(data, "time")
            metric = f"{prefix}_{bin_key}"
            y = col(data, metric).copy()

            # Empty radial bins are marked -1 in the output.
            y[y < 0] = np.nan

            ax.plot(t, y, linewidth=1.6, label=run_label)

        ax.set_title(bin_label)
        ax.set_xlim(left=0)
        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.grid(True, alpha=0.25)

    # Shared axis labels and a single legend keep the panels uncluttered.
    fig.supxlabel(r"Time, $t$ [simulation units]")
    fig.supylabel(ylabel)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="upper center", ncol=3,
        frameon=False, bbox_to_anchor=(0.5, 1.01)
    )

    fig.tight_layout(rect=(0.04, 0.04, 1, 0.93))
    save_figure(fig, filename)


for group_name, specs in GROUPS.items():
    plot_radial_bins(
        group_name, specs, "z_rms",
        r"Vertical RMS, $z_{\rm rms}(R)$ [length units]",
        f"{group_name}_vertical_structure_by_radius.pdf"
    )
    plot_radial_bins(
        group_name, specs, "A1",
        r"Azimuthal $m=1$ amplitude, $A_1(R)$",
        f"{group_name}_A1_by_radius.pdf",
        ylim=(0, 1)
    )
    plot_radial_bins(
        group_name, specs, "A2",
        r"Azimuthal $m=2$ amplitude, $A_2(R)$",
        f"{group_name}_A2_by_radius.pdf",
        ylim=(0, 1)
    )


# ---------------------------------------------------------------------

# 6. CENTRE-OF-MASS OFFSETS

# ---------------------------------------------------------------------



for group_name, specs in GROUPS.items():

    fig, ax = plt.subplots(figsize=(8.2, 5.0))

    for key, label in specs:

        data = DATA[key]

        ax.plot(

            col(data, "time"), col(data, "dz_cm_core"),

            linewidth=1.8, label=label

        )

    ax.set_xlabel(r"Time, $t$ [simulation units]")

    ax.set_ylabel(r"Vertical COM offset, $z_{\rm CM}-z_{\rm core}$ "

                  r"[length units]")

    ax.set_xlim(left=0)

    ax.grid(True, alpha=0.25)

    ax.legend(frameon=False)

    fig.tight_layout()

    save_figure(fig, f"{group_name}_vertical_COM_offset.pdf")



# ---------------------------------------------------------------------

# 7. CONSERVATION / NUMERICAL DIAGNOSTICS

# ---------------------------------------------------------------------



# Energy is the total simulation energy; Lz is calculated for primary

# disk particles. Plot fractional changes relative to the first output.

for group_name, specs in GROUPS.items():

    fig, axes = plt.subplots(2, 1, figsize=(8.2, 7.0), sharex=True)



    for key, label in specs:

        data = DATA[key]

        t = col(data, "time")

        E = col(data, "energy")

        Lz = col(data, "Lz_all")



        E0 = E[0]

        Lz0 = Lz[0]

        dE = (E - E0) / abs(E0) if E0 != 0 else np.full_like(E, np.nan)

        dLz = (Lz - Lz0) / abs(Lz0) if Lz0 != 0 else np.full_like(Lz, np.nan)



        axes[0].plot(t, dE, linewidth=1.5, label=label)

        axes[1].plot(t, dLz, linewidth=1.5, label=label)



    axes[0].set_ylabel(r"Fractional energy change, $(E-E_0)/|E_0|$")

    axes[1].set_ylabel(r"Fractional primary $L_z$ change")

    axes[1].set_xlabel(r"Time, $t$ [simulation units]")

    for ax in axes:

        ax.grid(True, alpha=0.25)

        ax.legend(frameon=False)

        ax.set_xlim(left=0)

    fig.tight_layout()

    save_figure(fig, f"{group_name}_conservation_diagnostics.pdf")



# ---------------------------------------------------------------------

# 8. LATE-TIME SUMMARY: MEAN AND STANDARD DEVIATION OVER t=70--100

# ---------------------------------------------------------------------



SUMMARY_METRICS = [

    "f_primary_present", "f_core",

    "R_rms_all", "z_rms_all", "R_rms_core", "z_rms_core",

    "deltaR_mean_abs", "deltaR_rms",

    "A1_all", "A2_all", "dcm_core",

    "z_rms_0_1", "z_rms_1_2", "z_rms_2_3", "z_rms_3_4.25",

    "A1_0_1", "A1_1_2", "A1_2_3", "A1_3_4.25",

    "A2_0_1", "A2_1_2", "A2_2_3", "A2_3_4.25",

]



summary_rows = []

for group_name, specs in GROUPS.items():

    for key, label in specs:

        data = DATA[key]

        t = col(data, "time")

        mask = (t >= SUMMARY_TMIN) & (t <= SUMMARY_TMAX)

        if not np.any(mask):

            raise ValueError(

                f"{FILES[key]} has no data between t={SUMMARY_TMIN} "

                f"and t={SUMMARY_TMAX}. Adjust SUMMARY_TMIN/TMAX."

            )



        row = {

            "comparison": group_name,

            "run": FILES[key],

            "curve_label": label,

            "time_interval": f"{SUMMARY_TMIN:g}-{SUMMARY_TMAX:g}",

        }

        for metric in SUMMARY_METRICS:

            values = col(data, metric)[mask]

            values = values[np.isfinite(values) & (values >= 0)]

            row[f"{metric}_mean"] = float(np.mean(values)) if values.size else np.nan

            row[f"{metric}_std"] = float(np.std(values, ddof=1)) if values.size > 1 else (

                0.0 if values.size == 1 else np.nan

            )

        summary_rows.append(row)



summary_path = OUTPUT_DIR / "late_time_summary.csv"

with summary_path.open("w", newline="") as f:

    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))

    writer.writeheader()

    writer.writerows(summary_rows)



# Summary plots show mean +/- temporal standard deviation in the selected

# late-time interval. This is temporal variation, not run-to-run uncertainty.

SUMMARY_PLOT_METRICS = [

    ("R_rms_all", r"$R_{\rm rms}$ [length units]", "summary_R_rms_all"),

    ("z_rms_all", r"$z_{\rm rms}$ [length units]", "summary_z_rms_all"),

    ("f_core", r"$f_{\rm core}$", "summary_core_fraction"),

    ("deltaR_mean_abs", r"$\langle|\Delta R|\rangle$ [length units]",

     "summary_mean_abs_migration"),

    ("A1_all", r"$A_1$", "summary_A1"),

    ("A2_all", r"$A_2$", "summary_A2"),

]



for group_name, specs in GROUPS.items():

    fig, axes = plt.subplots(2, 3, figsize=(12, 7.2))

    axes = axes.ravel()



    for ax, (metric, ylabel, _) in zip(axes, SUMMARY_PLOT_METRICS):

        means, stds, labels = [], [], []

        for key, label in specs:

            data = DATA[key]

            t = col(data, "time")

            mask = (t >= SUMMARY_TMIN) & (t <= SUMMARY_TMAX)

            values = col(data, metric)[mask]

            values = values[np.isfinite(values) & (values >= 0)]

            means.append(np.mean(values) if values.size else np.nan)

            stds.append(np.std(values, ddof=1) if values.size > 1 else 0.0)

            labels.append(label)



        x = np.arange(len(labels))

        ax.errorbar(x, means, yerr=stds, fmt="o", capsize=4,

                    markersize=6, linewidth=1.4)

        ax.set_xticks(x)

        ax.set_xticklabels(labels, rotation=15, ha="right")

        ax.set_ylabel(ylabel)

        ax.grid(True, axis="y", alpha=0.25)



    fig.suptitle(

        f"{group_name.replace('_', ' ').title()}: late-time mean "

        f"and temporal standard deviation "

        f"({SUMMARY_TMIN:g} <= t <= {SUMMARY_TMAX:g})"

    )

    fig.tight_layout()

    save_figure(fig, f"{group_name}_late_time_summary.pdf")



print(f"Loaded {len(DATA)} data files.")

print(f"Created PDF figures and summary CSV in: {OUTPUT_DIR}")

print("Reminder: late-time standard deviations describe temporal variation,")

print("not independent-run statistical uncertainties.")


Loaded 7 data files.
Created PDF figures and summary CSV in: \\wsl.localhost\Ubuntu-24.04\home\noro113\rebound\examples\selfgravity_disc\figures
Reminder: late-time standard deviations describe temporal variation,
not independent-run statistical uncertainties.
